# ML-02: Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

My research question and provisional lane (Lane 2, Refresh / Content Opportunity Scoring), backed by three numbers computed from the starter dataset.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring** (provisional, I can still change it until the end of Week 4).

In ML-01 I saw that a learned model put far more truly declining pages in its top 50 than a hand-written rule did (0.74 vs 0.24 precision@50), and that a rule that looked great in-sample lost most of its edge on clients it had never seen. I also saw that declining pages in this sample tend to be *younger*, not older, which goes against the usual "old content needs a refresh" belief. That makes "which pages should an editor look at first?" a question where the obvious rule is not enough and the data has something to say. It is also the lane closest to a decision someone actually makes every week, so the output can be judged by whether it would change what gets worked on.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sohamrathi17/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].eq("down")
print(f"{len(df):,} pages | {df['client_id'].nunique()} clients | declining rate {df['is_declining'].mean():.1%}")

30,000 pages | 32 clients | declining rate 54.2%


## 2. The question: decision, action, cost of a wrong call

**Search question:** for each client, which pages are losing search visibility and are worth an editor's review *first*?

| | |
|---|---|
| **Unit of analysis** | One page (content item), ranked within its own client |
| **Decision it improves** | Which K pages an editor reviews this week, out of hundreds per client |
| **Who acts** | The content editor / SEO lead responsible for that client's site |
| **Action** | Open the page and choose one of: refresh, expand, protect (leave it alone, it is fine), or monitor |
| **Output** | A ranked review queue per client: score, a short reason code (e.g. "visible, losing impressions, page 1"), a suggested action, and a confidence label |
| **Metric (named now, before any model)** | Precision@K per client against an observed decline label, plus the share of lost impressions captured in the top K |

**Cost of a wrong call.** A false positive (flagging a page that is fine) wastes an editor's hours, and editing a healthy page can even hurt it. A false negative (missing a page that is sliding) means lost clicks keep piling up unnoticed. Because the loss is very concentrated in a few pages (see section 3), missing one big page costs more than reviewing a couple of wrong ones, so the queue should weigh *how much* visibility is at risk, not only *whether* a page is declining.

**Why data / ML can help at all (and why this is not just "train a model").** The decision is a ranking under limited editor time, so a yes/no prediction is not the goal; a good order is. The simple rules people reach for do not fit this data well (the stale-page rule barely applies, see section 3), and the signals that matter (visibility, position, age, update history, engagement) interact in ways that are hard to hand-write. But a model only earns its place if it beats a transparent baseline on held-out clients. So the plan is: frame first, build the baseline rule, then test whether a model adds anything, validated with client-holdout splits so no client's pages appear in both train and test.

In [2]:
# How big is the review problem? Pages per client vs. a realistic weekly review budget.
pages_per_client = df.groupby("client_id").size()
declining_per_client = df.groupby("client_id")["is_declining"].sum()
print(f"Pages per client:     median {pages_per_client.median():.0f}  (range {pages_per_client.min()} to {pages_per_client.max()})")
print(f"Declining per client: median {declining_per_client.median():.0f}")
print("If an editor can review ~20 pages a week, the order of the queue is the whole decision.")

Pages per client:     median 567  (range 3 to 7008)
Declining per client: median 216
If an editor can review ~20 pages a week, the order of the queue is the whole decision.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV (computed below) that make this lane worth the next 7 weeks:

1. **About half of all pages are declining (54.2%, 16,262 pages).** A "declining or not" flag alone cannot set priorities; it just hands the editor a list of thousands. The value is in the *order*.
2. **The loss is highly concentrated.** The top 1% of pages by lost impressions (last 30 days vs the 30 before) account for about a third of all lost impressions, and the top 10% for over three quarters. Getting the top of the queue right matters far more than getting the whole list right, which is exactly what precision@K measures.
3. **The obvious "stale page" rule barely applies here.** Only 174 pages (0.6%) went 180+ days without an update, and those decline *less* often than average (47% vs 54%). The intuitive rule points at the wrong pages, which is a good reason to look at the data instead of assuming.

In [3]:
# 1) Base rate of the decline label
print(f"1) Declining pages: {df['is_declining'].sum():,} of {len(df):,} = {df['is_declining'].mean():.1%}")

# 2) How concentrated is the lost visibility?
lost = (df["impressions_prev_30d"] - df["impressions_last_30d"]).clip(lower=0).sort_values(ascending=False)
print(f"\n2) Total impressions lost (last 30d vs prev 30d): {lost.sum():,.0f}")
for share in (0.01, 0.05, 0.10):
    top_n = int(len(lost) * share)
    print(f"   top {share:>4.0%} of pages ({top_n:>5,}) hold {lost.iloc[:top_n].sum() / lost.sum():.1%} of the loss")

# 3) Does the 'stale page' rule (180+ days since last update) find the decliners?
stale = df["days_since_last_update"] >= 180
print(f"\n3) Stale pages (180+ days since update): {stale.sum()} ({stale.mean():.1%} of all pages)")
print(f"   declining rate, stale:     {df.loc[stale, 'is_declining'].mean():.1%}")
print(f"   declining rate, not stale: {df.loc[~stale, 'is_declining'].mean():.1%}")
print("\nDeclining rate by freshness tier:")
print(df.groupby("freshness_tier")["is_declining"].agg(pages="size", declining_rate="mean").round(3).to_string())

1) Declining pages: 16,262 of 30,000 = 54.2%

2) Total impressions lost (last 30d vs prev 30d): 15,456,445
   top   1% of pages (  300) hold 31.6% of the loss
   top   5% of pages (1,500) hold 63.3% of the loss
   top  10% of pages (3,000) hold 78.4% of the loss

3) Stale pages (180+ days since update): 174 (0.6% of all pages)
   declining rate, stale:     47.1%
   declining rate, not stale: 54.2%

Declining rate by freshness tier:
                pages  declining_rate
freshness_tier                       
0-30            20480           0.511
181+              174           0.471
31-90             175           0.589
91-180           9171           0.611


## 4. Careful words: what I can and can't claim

**What my work will be able to say:**
- *Observed:* which pages lost visibility in this data, and which signals were associated with that.
- *Directional:* patterns like "younger, visible pages decline more often in this sample", stated with their sample sizes.
- *Decision-support:* a ranked queue that, on held-out clients, puts more truly declining, high-impact pages near the top than a simple rule does. It helps an editor choose where to look; the editor still decides.

**What it will never claim:**
- That it predicts or reverse-engineers Google's ranking algorithm.
- That refreshing a page *causes* recovery. This data is observational; there is no experiment, so a declining page getting refreshed and recovering would not prove the refresh did it.
- That "declining" means a refresh will pay off. Some declines are seasonal or follow demand, and the right action can be "monitor".

**Known limits of the label (checked below).** The starter label is `trend_direction == "down"`: impressions in the last 30 days are more than 20% below the 30 days before. That is an observed outcome, but it sits in the *same window* as the 90-day totals, so a starter model is describing a trend, not forecasting one. Pages with zero impressions in the previous 30 days (11% of the sample) have no trend to measure, so they can never count as declining. And for small pages, a "down" can be a swing of just a few impressions: the decline *rate* is about the same for small and big pages (~60%), but a 25% drop on a 40-impression page is not worth an editor's hour, so the queue needs a volume floor. For the capstone I plan to build a proper future-window label from the warehouse daily table (features from earlier months, outcome from a later one), and `trend_direction` / `trend_pct` are never features.

In [4]:
# Label limits: how many pages have no previous-window baseline, and how noisy is 'down' for tiny pages?
no_prev = (df["impressions_prev_30d"] == 0).sum()
print(f"Pages with 0 impressions in the previous 30 days (no trend to measure): {no_prev:,} ({no_prev / len(df):.1%})")

small = df["impressions_prev_30d"].between(1, 49)
print(f"Pages with 1-49 previous-window impressions: {small.sum():,}; declining rate {df.loc[small, 'is_declining'].mean():.1%}")
big = df["impressions_prev_30d"] >= 1000
print(f"Pages with 1,000+ previous-window impressions: {big.sum():,}; declining rate {df.loc[big, 'is_declining'].mean():.1%}")
print("Similar decline rates, but a drop on a tiny page is a few impressions. A volume floor keeps the queue on changes big enough to matter.")

Pages with 0 impressions in the previous 30 days (no trend to measure): 3,388 (11.3%)
Pages with 1-49 previous-window impressions: 6,363; declining rate 59.8%
Pages with 1,000+ previous-window impressions: 8,030; declining rate 59.0%
Similar decline rates, but a drop on a tiny page is a few impressions. A volume floor keeps the queue on changes big enough to matter.


## Quality checks

Confirmed for this notebook:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (all outputs shown are from a full run)
- [x] No client names, URLs, or private queries anywhere (only pseudonymized aggregate numbers)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to the repo under `work/notebooks/`